# Covariate Augmentation for Reconstructed IPD

In this notebook, we augment reconstructed individual patient data (IPD) obtained from KM-GPT with baseline covariates derived from auxiliary trial summaries.

The reconstructed IPD contain survival times, censoring indicators, and treatment assignment, but lack individual-level baseline characteristics. To enable causal transport and target-population weighting, we construct synthetic covariates consistent with reported trial-level summaries.

This notebook:
1. loads the harmonized reconstructed IPD
2. defines auxiliary trial-level summaries
3. generates individual-level covariates consistent with these summaries
4. prepares the dataset for downstream weighting and Bayesian nonparametric analysis

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# For reproducibility
np.random.seed(733)

# Paths
DATA_DIR = Path("../data/processed")

# Load reconstructed IPD
df_real = pd.read_csv(DATA_DIR / "real_kmgpt_ipd_harmonized.csv")

print("Loaded df_real:", df_real.shape)
df_real.head()

Loaded df_real: (776, 7)


,trial_id,subgroup,time,event,arm_label,curve,treatment
0,KEYNOTE-181,high_pdl1,0.731707,1,Chemotherapy,0,0
1,KEYNOTE-181,high_pdl1,0.890244,0,Chemotherapy,0,0
2,KEYNOTE-181,high_pdl1,1.048780,1,Chemotherapy,0,0
3,KEYNOTE-181,high_pdl1,1.243902,1,Chemotherapy,0,0
4,KEYNOTE-181,high_pdl1,1.365854,1,Chemotherapy,0,0


## Auxiliary Summaries

Drawing on Population baseline characteristics from the KEYNOTE, ATTRACTION, and ESCORT Trials, we will save them as a CSV each so we can extract more if needed.

In [2]:
# Attraction 3 (disregard high/low subgroup for now)
import pandas as pd
from pathlib import Path

# Build full table (long format, easier to use later)
attraction3_table1 = pd.DataFrame({
    "variable": [
        "age_median",
        "male",
        "asian",
        "ecog_0",
        "ecog_1",
        "recurrent_yes",
        "stage_II_III",
        "stage_IV",
        "metastasis_ge2",
        "pd_l1_ge10",
        "smoking_current",
    ],
    "nivolumab": [
        64,
        0.85,
        0.96,
        0.48,
        0.52,
        0.49,
        0.07,
        0.88,
        0.58,
        0.30,
        0.10,
    ],
    "chemotherapy": [
        67,
        0.89,
        0.96,
        0.51,
        0.49,
        0.43,
        0.11,
        0.83,
        0.56,
        0.27,
        0.14,
    ],
})

# Add metadata
attraction3_table1["trial_id"] = "ATTRACTION-3"

attraction3_table1["subgroup"] = "overall"

# Save
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

attraction3_table1.to_csv(
    output_dir / "attraction3_table1_full.csv",
    index=False
)

print("Saved ATTRACTION-3 Table 1 to processed/")
attraction3_table1

Saved ATTRACTION-3 Table 1 to processed/


,variable,nivolumab,chemotherapy,trial_id,subgroup
0,age_median,64.00,67.00,ATTRACTION-3,overall
1,male,0.85,0.89,ATTRACTION-3,overall
2,asian,0.96,0.96,ATTRACTION-3,overall
3,ecog_0,0.48,0.51,ATTRACTION-3,overall
4,ecog_1,0.52,0.49,ATTRACTION-3,overall
5,recurrent_yes,0.49,0.43,ATTRACTION-3,overall
6,stage_II_III,0.07,0.11,ATTRACTION-3,overall
7,stage_IV,0.88,0.83,ATTRACTION-3,overall
8,metastasis_ge2,0.58,0.56,ATTRACTION-3,overall
9,pd_l1_ge10,0.30,0.27,ATTRACTION-3,overall


In [3]:
# ESCORT
import pandas as pd
from pathlib import Path

escort_table1 = pd.DataFrame({
    "variable": [
        "age_median",
        "male",
        "ecog_0",
        "ecog_1",
        "metastasis_ge2",
        "liver_met",
        "lung_met",
        "bone_met",
        "lymph_node_met",
        "pd_l1_ge10",
        "surgery",
        "radiotherapy",
        "first_line_chemo",
    ],
    "camrelizumab": [
        60,
        0.91,
        0.20,
        0.80,
        0.58,
        0.24,
        0.47,
        0.13,
        0.83,
        0.11,
        0.50,
        0.68,
        0.96,
    ],
    "chemotherapy": [
        60,
        0.87,
        0.20,
        0.80,
        0.62,
        0.21,
        0.43,
        0.13,
        0.88,
        0.16,
        0.48,
        0.63,
        0.93,
    ],
})

escort_table1["trial_id"] = "ESCORT"

# save
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

escort_table1.to_csv(
    output_dir / "escort_table1_full.csv",
    index=False
)

print("Saved ESCORT Table 1")
escort_table1

Saved ESCORT Table 1


,variable,camrelizumab,chemotherapy,trial_id
0,age_median,60.00,60.00,ESCORT
1,male,0.91,0.87,ESCORT
2,ecog_0,0.20,0.20,ESCORT
3,ecog_1,0.80,0.80,ESCORT
4,metastasis_ge2,0.58,0.62,ESCORT
5,liver_met,0.24,0.21,ESCORT
6,lung_met,0.47,0.43,ESCORT
7,bone_met,0.13,0.13,ESCORT
8,lymph_node_met,0.83,0.88,ESCORT
9,pd_l1_ge10,0.11,0.16,ESCORT


In [4]:
# Keynote (Note: Table 1 is for the overall KEYNOTE-181 ITT population, while your KM-GPT dataset is for the high PD-L1 subgroup. That is still usable for auxiliary information)
import pandas as pd
from pathlib import Path

keynote181_table1 = pd.DataFrame({
    "variable": [
        "age_median",
        "age_ge65",
        "male",
        "asia_region",
        "ecog_0",
        "ecog_1",
        "ecog_2",
        "squamous_histology",
        "adenocarcinoma",
        "pd_l1_ge10",
        "pd_l1_lt10",
        "prior_adjuvant_neoadjuvant",
        "metastatic_stage",
        "locally_advanced_stage",
        "prior_therapies_0",
        "prior_therapies_1",
        "prior_therapies_ge2",
    ],
    "pembrolizumab": [
        63.0,
        0.443,
        0.869,
        0.385,
        0.401,
        0.596,
        0.003,
        0.631,
        0.369,
        0.341,
        0.640,
        0.102,
        0.924,
        0.076,
        0.006,
        0.965,
        0.029,
    ],
    "chemotherapy": [
        62.0,
        0.424,
        0.863,
        0.389,
        0.369,
        0.627,
        0.003,
        0.646,
        0.354,
        0.366,
        0.624,
        0.102,
        0.911,
        0.089,
        0.000,
        0.987,
        0.013,
    ],
})

keynote181_table1["trial_id"] = "KEYNOTE-181"

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

keynote181_table1.to_csv(
    output_dir / "keynote181_table1_full.csv",
    index=False
)

print("Saved KEYNOTE-181 Table 1")
keynote181_table1

Saved KEYNOTE-181 Table 1


,variable,pembrolizumab,chemotherapy,trial_id
0,age_median,63.000,62.000,KEYNOTE-181
1,age_ge65,0.443,0.424,KEYNOTE-181
2,male,0.869,0.863,KEYNOTE-181
3,asia_region,0.385,0.389,KEYNOTE-181
4,ecog_0,0.401,0.369,KEYNOTE-181
5,ecog_1,0.596,0.627,KEYNOTE-181
6,ecog_2,0.003,0.003,KEYNOTE-181
7,squamous_histology,0.631,0.646,KEYNOTE-181
8,adenocarcinoma,0.369,0.354,KEYNOTE-181
9,pd_l1_ge10,0.341,0.366,KEYNOTE-181


In [8]:
# Combine to single ux sum table
combined_aux = pd.DataFrame({
    "trial_id": [
        "KEYNOTE-181",
        "ESCORT",
        "ATTRACTION-3",
        "ATTRACTION-3",
    ],
    "subgroup": [
        "high_pdl1",
        "high_pdl1",
        "low_pdl1",
        "high_pdl1",
    ],
    "age_median": [
        63.0,   # KEYNOTE-181 overall Table 1
        60.0,   # ESCORT overall Table 1
        65.0,   # ATTRACTION-3 overall Table 1
        65.0,   # ATTRACTION-3 overall Table 1
    ],
    "male_rate": [
        0.869,
        0.91,
        0.85,
        0.85,
    ],
    "ecog0_rate": [
        0.401,
        0.20,
        0.48,
        0.48,
    ],
    "metastatic_rate": [
        0.924,
        0.58,   # using organs>=2 as a severity proxy from ESCORT
        0.88,
        0.88,
    ],
    "biomarker_rate": [
        0.70,   # high PD-L1 proxy
        0.85,   # ESCORT high PD-L1 only; use a high value
        0.30,   # low PD-L1
        0.70,   # high PD-L1
    ],
})
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

combined_aux.to_csv(
    output_dir / "combined_auxiliary_summaries.csv",
    index=False
)

print("Saved combined auxiliary summaries.")
combined_aux

Saved combined auxiliary summaries.


,trial_id,subgroup,age_median,male_rate,ecog0_rate,metastatic_rate,biomarker_rate
0,KEYNOTE-181,high_pdl1,63.0,0.869,0.401,0.924,0.70
1,ESCORT,high_pdl1,60.0,0.910,0.200,0.580,0.85
2,ATTRACTION-3,low_pdl1,65.0,0.850,0.480,0.880,0.30
3,ATTRACTION-3,high_pdl1,65.0,0.850,0.480,0.880,0.70


In [6]:
df_aug = df_real.merge(
    combined_aux,
    on=["trial_id", "subgroup"],
    how="left"
)

print(df_aug.shape)
df_aug.head()

(776, 12)


,trial_id,subgroup,time,event,arm_label,curve,treatment,age_median,male_rate,ecog0_rate,metastatic_rate,biomarker_rate
0,KEYNOTE-181,high_pdl1,0.731707,1,Chemotherapy,0,0,63.0,0.869,0.401,0.924,0.7
1,KEYNOTE-181,high_pdl1,0.890244,0,Chemotherapy,0,0,63.0,0.869,0.401,0.924,0.7
2,KEYNOTE-181,high_pdl1,1.048780,1,Chemotherapy,0,0,63.0,0.869,0.401,0.924,0.7
3,KEYNOTE-181,high_pdl1,1.243902,1,Chemotherapy,0,0,63.0,0.869,0.401,0.924,0.7
4,KEYNOTE-181,high_pdl1,1.365854,1,Chemotherapy,0,0,63.0,0.869,0.401,0.924,0.7


In [7]:
df_aug[["trial_id", "subgroup", "age_median", "biomarker_rate"]].drop_duplicates()

,trial_id,subgroup,age_median,biomarker_rate
0,KEYNOTE-181,high_pdl1,63.0,0.70
166,ESCORT,high_pdl1,60.0,0.85
357,ATTRACTION-3,low_pdl1,65.0,0.30
573,ATTRACTION-3,high_pdl1,65.0,0.70


## Individual Simulation

We’ll start simple:
- age ~ Normal(mean, sd)
- biomarker ~ Bernoulli(rate)
- stage ~ Normal(mean, sd)

In [9]:
from src.covariates.augmentation import (
    simulate_covariates_from_aux,
    summarize_simulated_covariates,
)

# Load auxiliary summaries
aux = pd.read_csv("../data/processed/combined_auxiliary_summaries.csv")

# Merge onto harmonized real-data IPD
df_aug = df_real.merge(
    aux,
    on=["trial_id", "subgroup"],
    how="left",
)

# Simulate individual-level covariates
df_cov = simulate_covariates_from_aux(
    df_aug,
    age_sd=8.0,
    stage_sd=0.25,
    random_state=733,
)

print("Augmented dataset shape:", df_cov.shape)
df_cov.head()

Augmented dataset shape: (776, 18)


,trial_id,subgroup,time,event,arm_label,curve,treatment,age_median,male_rate,ecog0_rate,metastatic_rate,biomarker_rate,age,male,ecog0,metastatic,biomarker,stage
0,KEYNOTE-181,high_pdl1,0.731707,1,Chemotherapy,0,0,63.0,0.869,0.401,0.924,0.7,56.369120,1,0,0,0,1.165542
1,KEYNOTE-181,high_pdl1,0.890244,0,Chemotherapy,0,0,63.0,0.869,0.401,0.924,0.7,68.148584,1,1,1,0,1.822921
2,KEYNOTE-181,high_pdl1,1.048780,1,Chemotherapy,0,0,63.0,0.869,0.401,0.924,0.7,58.035384,1,0,1,1,2.330231
3,KEYNOTE-181,high_pdl1,1.243902,1,Chemotherapy,0,0,63.0,0.869,0.401,0.924,0.7,61.334523,1,1,1,1,1.968789
4,KEYNOTE-181,high_pdl1,1.365854,1,Chemotherapy,0,0,63.0,0.869,0.401,0.924,0.7,73.194077,1,0,1,1,2.342307


In [10]:
cov_summary = summarize_simulated_covariates(df_cov)
cov_summary

,trial_id,subgroup,age_mean,male_rate,ecog0_rate,metastatic_rate,biomarker_rate,stage_mean,n
0,ATTRACTION-3,high_pdl1,64.284648,0.837438,0.492611,0.886700,0.783251,2.277758,203
1,ATTRACTION-3,low_pdl1,64.536380,0.884259,0.509259,0.851852,0.296296,2.149919,216
2,ESCORT,high_pdl1,60.579461,0.869110,0.178010,0.518325,0.821990,2.107681,191
3,KEYNOTE-181,high_pdl1,62.156532,0.867470,0.373494,0.933735,0.704819,2.331718,166


In [11]:
# naive pooled effect
from src.causal.estimands import pooled_survival_difference

pooled_cov = pooled_survival_difference(df_cov, t0=12.0)
pooled_cov

,t0,S0,S1,Delta
0,12.0,0.280376,0.449162,0.168785


In [12]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

df_cov.to_csv(output_dir / "real_kmgpt_ipd_with_covariates.csv", index=False)
cov_summary.to_csv(output_dir / "real_kmgpt_covariate_summary.csv", index=False)

print("Saved augmented real-data IPD with covariates.")

Saved augmented real-data IPD with covariates.
